## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


## <center>Extra 4: Regresori ML: exemple de cod</center>


**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. univ. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/04-regresie/extra-04-regresori-ml.ipynb)

In [1]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/04-regresie
sys.path.insert(0, os.path.abspath("../.."))

> **Notă:** fiecare model este căutat cu `GridSearchCV` în 5 repetări; notebook-ul complet rulează câteva minute. Pentru o căutare mai rapidă, cu buget de timp, vezi Optuna în [modulul 6](../06-evaluare-optimizare/README.md).

Pentru evaluarea regresorilor scikitlearn propune metricile de la sectiunea Regression de aici:

https://scikit-learn.org/stable/modules/model_evaluation.html



Daca adevaratul model este un model liniar, atunci media reziduurilor ar fi zero.

Daca media reziduurilor este departe de 0, atunci modelul liniar nu e cel dorit.

Când regresori liniari și neliniari sunt evaluati, putem considera scorul **explained_variance** ca fiind o generalizare a lui **$R^2$**.

Metrici uzuale folosite pentru antrenare în scopul prognozării sunt **MAE** și **MAPE** discutate și la cursul cu regresia liniară.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

## Setul de date

<a href="https://www.kaggle.com/code/prasadperera/the-boston-housing-dataset">The Boston Housing Dataset</a>

The Boston Housing Dataset is a derived from information collected by the U.S. Census Service concerning housing in the area of Boston MA. The following describes the dataset columns:

- CRIM - per capita crime rate by town
- ZN - proportion of residential land zoned for lots over 25,000 sq.ft.
- INDUS - proportion of non-retail business acres per town.
- CHAS - Charles River dummy variable (1 if tract bounds river; 0 otherwise)
- NOX - nitric oxides concentration (parts per 10 million)
- RM - average number of rooms per dwelling
- AGE - proportion of owner-occupied units built prior to 1940
- DIS - weighted distances to five Boston employment centres
- RAD - index of accessibility to radial highways
- TAX - full-value property-tax rate per USD10,000
- PTRATIO - pupil-teacher ratio by town
- B - 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
- LSTAT - % lower status of the population
- MEDV - Median value of owner-occupied homes in USD1000's

In [3]:
bostonhousing = pd.read_csv('./date/boston_housing.csv')
bostonhousing = bostonhousing.dropna(how='any')
bostonhousing.info()

target_col = 'MEDV'

<class 'pandas.DataFrame'>
Index: 401 entries, 0 to 449
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     401 non-null    float64
 1   ZN       401 non-null    float64
 2   INDUS    401 non-null    float64
 3   CHAS     401 non-null    float64
 4   NOX      401 non-null    float64
 5   RM       401 non-null    float64
 6   age      401 non-null    float64
 7   DIS      401 non-null    float64
 8   RAD      401 non-null    int64  
 9   IMPOZIT  401 non-null    float64
 10  PTRATIO  401 non-null    float64
 11  B        401 non-null    float64
 12  LSTAT%   401 non-null    float64
 13  MEDV     401 non-null    float64
dtypes: float64(13), int64(1)
memory usage: 47.0 KB


In [4]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

fig, axs = plt.subplots(ncols=7, nrows=2, figsize=(20, 10))
index = 0
axs = axs.flatten()
for k,v in bostonhousing.items():
    sns.boxplot(y=k, data=bostonhousing, ax=axs[index])
    index += 1
plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=5.0)

In [5]:
plt.figure(figsize=(20, 10))
sns.heatmap(bostonhousing.corr().abs(),  annot=True)

<Axes: >

Din matricea de corelație, vedem că IMPOZIT și RAD sunt caracteristici foarte corelate. Coloanele LSTAT, INDUS, RM, IMPOZIT, NOX, PTRAIO au un scor de corelare peste 0,5 cu MEDV, ceea ce este un bun indiciu ca să îi utilizăm ca predictori. Să punem aceste coloane în raport cu MEDV.

In [6]:
from sklearn import preprocessing
# Let's scale the columns before plotting them against MEDV
min_max_scaler = preprocessing.MinMaxScaler()
column_sels = ['LSTAT%', 'INDUS', 'NOX', 'PTRATIO', 'RM', 'IMPOZIT', 'DIS', 'age']
x = bostonhousing.loc[:,column_sels]
y = bostonhousing['MEDV']
x = pd.DataFrame(data=min_max_scaler.fit_transform(x), columns=column_sels)
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(20, 10))
index = 0
axs = axs.flatten()
for i, k in enumerate(column_sels):
    sns.regplot(y=y, x=x[k], ax=axs[i])
plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=5.0)

## Exemplu de antrenare regresor kNN cu GridSearch

In [7]:
from sklearn.neighbors import KNeighborsRegressor

In [8]:
def antrenament_scikitlearn_knnr(X,y):
    
    # instațiem un model
    knnr = KNeighborsRegressor()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid = {
                    'n_neighbors': list(range(2,20))
                 }
    
    # folosim  antrenare cu validare încrucișată
    CV_knnr = GridSearchCV(estimator=knnr, 
                          param_grid=param_grid, 
                          scoring = 'neg_mean_absolute_error', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_knnr.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_knnr.best_params_, CV_knnr.best_estimator_,CV_knnr.cv_results_

In [9]:
# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(bostonhousing.drop(columns=target_col),  # ținta NU este predictor
 
                                                        bostonhousing[target_col], 
                                                        test_size=0.3) # 70% training / 30% test
    
    #params,model = antrenament_scikitlearn_rfc(X_train.to_numpy(), y_train.to_numpy())
    params,model,full_cv_results = antrenament_scikitlearn_knnr(X_train, y_train)
    
    df = pd.DataFrame()
    df['n_neighbors'] = [p['n_neighbors']  for p in full_cv_results['params']]
    df['mean_test_score'] = abs(full_cv_results['mean_test_score'])

    # Extract the date and close price columns
    vecini = df['n_neighbors']
    mean_test_score = df['mean_test_score']

    # Create a line plot
    plt.plot(vecini, mean_test_score)

    # Show the plot
    plt.show()
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'mae': mae(y_true = y_test, y_pred=y_test_pred),
        'mape': f"{100*mape(y_true = y_test, y_pred=y_test_pred):.3f}%",
        'parametrii': params}
        )

Iteration 0


Iteration 1
Iteration 2
Iteration 3


Iteration 4


In [10]:
print("Average MAE knnr:", sum([r['mae'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Average MAE knnr: 5.207920110192838


,mae,mape,parametrii
0,4.874174,19.674%,{'n_neighbors': 4}
1,4.488430,17.408%,{'n_neighbors': 2}
2,5.711983,27.529%,{'n_neighbors': 2}
3,5.544077,20.819%,{'n_neighbors': 3}
4,5.420937,22.447%,{'n_neighbors': 3}


## Exemplu de antrenare arbore de regresie cu GridSearch

In [11]:
from sklearn.tree import DecisionTreeRegressor

In [12]:
def antrenament_scikitlearn_dtr(X,y):
    
    # instațiem un model
    dtr = DecisionTreeRegressor()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    # Hyper parameters range intialization for tuning 

    # 162 de combinații (inițial 1800, care rulau aproape jumătate de oră)
    param_grid={
                "splitter":["best"],
                "max_depth" : [3,5,7],
               "min_samples_leaf":[3,5,7],
               "min_weight_fraction_leaf":[0.0,0.1,0.2],
               "max_features":["sqrt",None],
               "max_leaf_nodes":[None,10,50] 
               }
    
    # folosim  antrenare cu validare încrucișată
    CV_dtr = GridSearchCV(estimator=dtr, 
                          param_grid=param_grid, 
                          scoring = 'neg_mean_absolute_error', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_dtr.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_dtr.best_params_, CV_dtr.best_estimator_

In [13]:
# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(bostonhousing.drop(columns=target_col),  # ținta NU este predictor
 
                                                        bostonhousing[target_col], 
                                                        test_size=0.3) # 70% training / 30% test
    
    #params,model = antrenament_scikitlearn_rfc(X_train.to_numpy(), y_train.to_numpy())
    params,model = antrenament_scikitlearn_dtr(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'mae': mae(y_true = y_test, y_pred=y_test_pred),
        'mape': f"{100*mape(y_true = y_test, y_pred=y_test_pred):.3f}%",
        'parametrii': params}
        )

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


In [14]:
print("Average MAE:", sum([r['mae'] for r in results_CV_paralel])/len(results_CV_paralel))
pd.options.display.max_colwidth = 200
pd.DataFrame(results_CV_paralel)

Average MAE: 2.9575608527142165


,mae,mape,parametrii
0,2.549260,10.336%,"{'max_depth': 7, 'max_features': None, 'max_leaf_nodes': 50, 'min_samples_leaf': 3, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}"
1,2.987173,12.225%,"{'max_depth': 5, 'max_features': None, 'max_leaf_nodes': 50, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}"
2,2.419205,11.042%,"{'max_depth': 7, 'max_features': None, 'max_leaf_nodes': 50, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}"
3,3.509142,13.374%,"{'max_depth': 5, 'max_features': None, 'max_leaf_nodes': None, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}"
4,3.323025,13.859%,"{'max_depth': 7, 'max_features': None, 'max_leaf_nodes': 50, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}"


## Exemplu de antrenare regresor Random Forest cu GridSearch

In [15]:
from sklearn.ensemble import RandomForestRegressor




In [16]:
bostonhousing

,CRIM,ZN,INDUS,CHAS,NOX,RM,age,DIS,RAD,IMPOZIT,PTRATIO,B,LSTAT%,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,9.96654,0.0,18.10,0.0,0.740,6.485,100.0,1.9784,24,666.0,20.2,386.73,18.85,15.4
446,6.28807,0.0,18.10,0.0,0.740,6.341,96.4,2.0720,24,666.0,20.2,318.01,17.79,14.9
447,9.92485,0.0,18.10,0.0,0.740,6.251,96.6,2.1980,24,666.0,20.2,388.52,16.44,12.6
448,9.32909,0.0,18.10,0.0,0.713,6.185,98.7,2.2616,24,666.0,20.2,396.90,18.13,14.1


In [17]:
def antrenament_scikitlearn_rfr(X,y):
    
    # instațiem un model
    rfr = RandomForestRegressor()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid = {
                    'n_estimators': [100, 300], #[100, 200, 500, 1000],
                    'max_features': ['sqrt','log2', int(round(X.shape[0]/3,0)) ], #cutuma pt regresie: 1/3 din numarul de atribute
                    'max_samples' : [0.1, 0.2, 0.3, 0.4] ,
                    #'criterion' :['gini', 'entropy']
                 }
    
    # folosim  antrenare cu validare încrucișată
    CV_rfr = GridSearchCV(estimator=rfr, 
                          param_grid=param_grid, 
                          scoring = 'neg_mean_absolute_error', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_rfr.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_rfr.best_params_, CV_rfr.best_estimator_



In [18]:
# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(bostonhousing.drop(columns=target_col),  # ținta NU este predictor
 
                                                        bostonhousing[target_col], 
                                                        test_size=0.3) # 70% training / 30% test
    
    #params,model = antrenament_scikitlearn_rfc(X_train.to_numpy(), y_train.to_numpy())
    params,model = antrenament_scikitlearn_rfr(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'mae': mae(y_true = y_test, y_pred=y_test_pred),
        'mape': f"{100*mape(y_true = y_test, y_pred=y_test_pred):.3f}%",
        'parametrii': params}
        )
    



Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


In [19]:
print("Average MAE:", sum([r['mae'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Average MAE: 2.4669867768595037


,mae,mape,parametrii
0,2.249928,9.875%,"{'max_features': 93, 'max_samples': 0.4, 'n_estimators': 300}"
1,2.828198,10.241%,"{'max_features': 93, 'max_samples': 0.4, 'n_estimators': 100}"
2,2.667904,13.656%,"{'max_features': 93, 'max_samples': 0.4, 'n_estimators': 300}"
3,2.442413,11.086%,"{'max_features': 93, 'max_samples': 0.4, 'n_estimators': 100}"
4,2.146490,10.040%,"{'max_features': 93, 'max_samples': 0.4, 'n_estimators': 300}"


# Exemplu de antrenare regresor XGBoost cu Grid Search pentru parametrii

### 1. implementarea din scikitlearn

In [20]:
from sklearn.ensemble import GradientBoostingRegressor
import time

In [21]:
def antrenament_scikitlearn_xgb_r(X,y):
    
    xgb = GradientBoostingRegressor()#random_state=42)
    param_grid = {
                    'n_estimators' : [10, 100, 300],
                    'learning_rate' : [0.001, 0.01, 0.1],
                    #'subsample' : [0.5, 0.7, 1.0],
                    #'max_depth' : [3, 7, 9]
                 }
    CV_xgb = GridSearchCV(estimator=xgb, 
                          param_grid=param_grid,
                          scoring = 'explained_variance',
                          cv= 10, 
                          n_jobs = 6)
    CV_xgb.fit(X, y)
    
    return CV_xgb.best_params_, CV_xgb.best_estimator_



Apelarea functiei de antrenament:

In [22]:
# 100 de iteratii 

tic = time.time()
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset into training set and test set
    X_train, X_test, y_train, y_test = train_test_split(bostonhousing.drop(columns=target_col),  # ținta NU este predictor
 bostonhousing[target_col], test_size=0.3) # 70% training and 30% test
    
    params,model = antrenament_scikitlearn_xgb_r(X_train, y_train)
    
    yhat = model.predict(X_test)
    error = abs(y_test-yhat.reshape(y_test.shape[0])).sum()/y_test.shape[0]
    res_dict = pd.Series(abs(y_test-yhat.reshape(y_test.shape[0]))).describe(
    percentiles = [0.75,.8,0.95,0.99]).to_dict()
    res_dict['mae'] = mae(y_true = y_test, y_pred=yhat)
    res_dict['mape'] = f"{100*mape(y_true = y_test, y_pred=yhat):.3f}%"
    res_dict['parametrii'] = params
    results_CV_paralel.append(res_dict)
    
toc = time.time()

print("Durata antrenament:", toc-tic)

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


Durata antrenament: 4.735814332962036


In [23]:
print("Average MAE xgboost:", sum([r['mae'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Average MAE xgboost: 2.2327150270159715


,count,mean,std,min,75%,80%,95%,99%,max,mae,mape,parametrii
0,121.0,2.260835,2.707579,0.079584,2.974164,3.619575,5.562772,8.095987,25.112067,2.260835,9.796%,"{'learning_rate': 0.1, 'n_estimators': 100}"
1,121.0,2.202304,2.253356,0.002681,2.905263,3.296546,5.771708,11.525976,14.144751,2.202304,8.893%,"{'learning_rate': 0.1, 'n_estimators': 100}"
2,121.0,2.557435,3.643755,0.033077,2.679679,2.962928,7.799308,15.486897,29.610367,2.557435,12.843%,"{'learning_rate': 0.1, 'n_estimators': 300}"
3,121.0,1.966557,2.150265,0.024800,2.747443,3.204923,6.977544,9.423093,11.387977,1.966557,8.801%,"{'learning_rate': 0.1, 'n_estimators': 300}"
4,121.0,2.176444,2.185111,0.060193,2.800354,3.008119,6.018607,10.421951,14.192449,2.176444,9.236%,"{'learning_rate': 0.1, 'n_estimators': 100}"


### 2. implementarea din xgboost

In [24]:
import xgboost as xgb
xgb.set_config(verbosity=0)


In [25]:
def antrenament_scikitlearn_xgboost_r(X,y):
    
    xgb_reg = xgb.XGBRegressor()#random_state=42)
    param_grid = {
                    'n_estimators' : [100, 300],
                    'eta' : [0.01, 0.1],
                    'max_depth' : [3, 6],
                    'subsample' : [0.7, 1.0],
                    'colsample_bytree':[0.7, 1.0],
                    'tree_method':['hist'] #typilcally fastest according to https://www.anyscale.com/blog/how-to-speed-up-xgboost-model-training
                    
                 }
    CV_xgb = GridSearchCV(estimator=xgb_reg, 
                          param_grid=param_grid,
                          scoring = 'explained_variance',
                          cv= 10, 
                          n_jobs = 6)
    CV_xgb.fit(X, y)
    
    return CV_xgb.best_params_, CV_xgb.best_estimator_

In [26]:
# cateva de iteratii 

tic = time.time()
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset into training set and test set
    X_train, X_test, y_train, y_test = train_test_split(bostonhousing.drop(columns=target_col),  # ținta NU este predictor
 bostonhousing[target_col], test_size=0.3) # 70% training and 30% test
    
    params,model = antrenament_scikitlearn_xgboost_r(X_train, y_train)
    
    yhat = model.predict(X_test)
    error = abs(y_test-yhat.reshape(y_test.shape[0])).sum()/y_test.shape[0]
    res_dict = pd.Series(abs(y_test-yhat.reshape(y_test.shape[0]))).describe(
    percentiles = [0.75,.8,0.95,0.99]).to_dict()
    res_dict['mae'] = mae(y_true = y_test, y_pred=yhat)
    res_dict['mape'] = f"{100*mape(y_true = y_test, y_pred=y_test_pred):.3f}%"
    res_dict['parametrii'] = params
    results_CV_paralel.append(res_dict)

toc = time.time()
print('Durata antrenament:', toc-tic)
print("Average MAE xgboost from xgboot:", sum([r['mae'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


Durata antrenament: 15.663517951965332
Average MAE xgboost from xgboot: 2.165202641132449


,count,mean,std,min,75%,80%,95%,99%,max,mae,mape,parametrii
0,121.0,2.232477,2.003871,0.001379,3.288875,3.676097,6.288132,8.085758,11.694843,2.232477,37.382%,"{'colsample_bytree': 0.7, 'eta': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7, 'tree_method': 'hist'}"
1,121.0,2.096203,2.162108,0.005712,2.823520,3.246941,6.948050,8.766806,10.491441,2.096203,38.031%,"{'colsample_bytree': 0.7, 'eta': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.7, 'tree_method': 'hist'}"
2,121.0,2.383575,3.104554,0.013768,3.081686,3.304301,7.732460,18.404193,20.387819,2.383575,38.658%,"{'colsample_bytree': 0.7, 'eta': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.7, 'tree_method': 'hist'}"
3,121.0,2.019878,2.216383,0.018313,2.771655,3.042880,4.928226,9.645508,17.101919,2.019878,34.419%,"{'colsample_bytree': 0.7, 'eta': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7, 'tree_method': 'hist'}"
4,121.0,2.093879,1.464602,0.029872,2.791726,3.056605,4.646820,6.680644,8.272034,2.093879,38.994%,"{'colsample_bytree': 0.7, 'eta': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7, 'tree_method': 'hist'}"
